<a href="https://colab.research.google.com/github/RomanPhys/External/blob/main/external_context_v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Внешний контекст, версия v7. Обучаемая память переходов

Зайцев предложил сначала выучить блок памяти отдельно и только потом вмешивать его в модель через гейт. Этот ноутбук проверяет ровно это на замороженном SASRec авторов.

Блок смотрит на переходы других пользователей, то есть на пары из состояния и следующего события. Для каждого запроса он берёт 200 переходов, похожих по косинусу состояний SASRec, только чужих и только завершённых раньше запроса, и переранжирует их выученной метрикой с выученным весом свежести. Значения в блоке это следующие события, а не состояния, как в статье. В начале обучения блок совпадает с памятью без обучения из прошлых версий.

Сравниваются такие варианты. Память без обучения, как в прошлых версиях. Блок, обученный отдельно, с гейтом lambda, это предложение Зайцева. Тот же блок с гейтом, который зависит от уверенности памяти. Блок, дообученный уже в смеси с замороженным SASRec. И контроль, тот же блок, у которого значения это состояния, как в статье, со смесью в вероятностях и со смесью в эмбеддинге.

Блок учится на ранних позициях валидационных пользователей. SASRec их не видел, как и тестовых, поэтому блок учится в тех же условиях, в которых потом работает. Последние пять позиций тех же пользователей идут на выбор гейта, а тестовые пользователи используются только для итоговой оценки.

Если на Drive осталась папка ext_work_v5 с выгруженными состояниями SASRec, GPU не нужен, и всё займёт около часа на процессоре. Если её нет, ноутбук заново обучит SASRec, тогда включи T4, это ещё около 10 минут на разбиение. Часть с Churn берёт кеш из ext_work_v6, если он там остался, иначе пересчитывает его минут за двадцать. Запусти все ячейки через Run all.

## Настройки

USE_DRIVE лучше оставить True. Тогда всё сохраняется в папку ext_work_v7 на Google Drive, и после обрыва сессии посчитанное пропустится. SPLITS задаёт разбиения пользователей для SASRec, как в v5. RUN_BANK включает ту же проверку на Churn в постановке статьи, с разбиением по клиентам и предсказанием следующего MCC.

In [ ]:
USE_DRIVE = True
SPLITS = [0, 1, 2]
RUN_BANK = True
BANK = "churn"
BANK_MAX_USERS = 12000

import os, json, glob, zipfile, shutil, subprocess, time
import numpy as np
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ext_work_v7"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
ROOT = os.path.dirname(WORK)
OLD5, OLD6 = f"{ROOT}/ext_work_v5", f"{ROOT}/ext_work_v6"
print("результаты будут в", WORK)

Mounted at /content/drive
результаты будут в /content/drive/MyDrive/ext_work_v7


## 1. Код

Когда появится кнопка выбора файла, загрузи external_context_v7.zip. Если архив уже лежит в папке ext_work_v7 на Drive, ноутбук возьмёт его оттуда.

In [ ]:
CODE = "/content/ext"
if not os.path.exists(f"{CODE}/code/run_learned_memory.py"):
    shutil.rmtree(CODE, ignore_errors=True)
    found = sorted(glob.glob(f"{WORK}/external_context_v*.zip"))
    if found:
        path = found[-1]
    else:
        from google.colab import files
        path = next(iter(files.upload()))
        if USE_DRIVE:
            shutil.copy(path, WORK)
    zipfile.ZipFile(path).extractall(CODE)
assert os.path.exists(f"{CODE}/code/run_learned_memory.py"), "это старый архив, нужен external_context_v7.zip"
print("код на месте", sorted(os.listdir(CODE)))
!cd {CODE}/code && python test_learned_memory.py


def run(script, W, log, **env):
    # каждый шаг в отдельном процессе с чистыми переменными EXT_
    e = {k: v for k, v in os.environ.items() if not k.startswith("EXT_")}
    e.update({k: str(v) for k, v in env.items()})
    e["EXT_WORK"] = W
    t0 = time.time()
    with open(f"{W}/{log}", "w") as f:
        p = subprocess.run(["python", script], env=e, stdout=f, stderr=subprocess.STDOUT, cwd=f"{CODE}/code")
    if p.returncode != 0:
        print(open(f"{W}/{log}").read()[-3000:])
        raise RuntimeError(f"{script} упал, хвост лога выше")
    print(f"    {script} готов за {time.time() - t0:.0f} с")


def show(script, W, *args, **env):
    e = {k: v for k, v in os.environ.items() if not k.startswith("EXT_")}
    e.update({k: str(v) for k, v in env.items()})
    e["EXT_WORK"] = W
    p = subprocess.run(["python", script, *args], env=e, capture_output=True, text=True, cwd=f"{CODE}/code")
    print(p.stdout)
    if p.returncode != 0:
        print(p.stderr[-3000:])


def fresh_states(p):
    try:
        return int(np.load(p)["export_version"]) == 2
    except Exception:
        return False

Saving external_context_v7.zip to external_context_v7.zip
код на месте ['README.md', 'code', 'colab', 'results', 'torch_patch']
метки в смеси: dW отн. ошибка 1.12e-09, dbeta +0.019702 против +0.019702, drho +1.338091 против +1.338091
метки отдельно: dW отн. ошибка 1.35e-09, dbeta +0.153561 против +0.153561, drho +2.712960 против +2.712960
состояния отдельно: dW отн. ошибка 6.60e-10, dbeta +0.332275 против +0.332275, drho -2.574399 против -2.574399
градиенты сходятся


# Часть A. SASRec авторов на MovieLens-1M

## A1. Состояния SASRec

Ноутбук ищет состояния, выгруженные в v5, в папке ext_work_v5 на Drive. Если их там нет, он учит SASRec заново тем же скриптом, что в v5, только без контрольных эпох и агрегаций авторов, они уже посчитаны.

In [ ]:
SRC, need_train = {}, []
for s in SPLITS:
    W = f"{WORK}/split{s}"
    os.makedirs(W, exist_ok=True)
    st_new, ca_new = f"{W}/sasrec_states.npz", f"{W}/cache_context.pkl"
    st_old, ca_old = f"{OLD5}/split{s}/sasrec_states.npz", f"{OLD5}/split{s}/cache_context.pkl"
    if fresh_states(st_new):
        SRC[s] = (st_new, ca_new)
    elif fresh_states(st_old):
        SRC[s] = (st_old, ca_old if os.path.exists(ca_old) else ca_new)
    else:
        SRC[s] = (st_new, ca_new)
        need_train.append(s)
    print(f"разбиение {s}:", "SASRec надо обучить заново" if s in need_train else f"состояния {SRC[s][0]}")

if need_train:
    import torch
    print("GPU есть" if torch.cuda.is_available() else "GPU нет, на процессоре SASRec будет учиться очень долго, лучше включи T4")
    %cd /content
    !test -d eca || git clone -q --depth 1 https://github.com/petrsokerin/External-Context-Aggregation.git eca
    !mkdir -p eca/recsys/src/data eca/recsys/data/movielens_1m
    !cp {CODE}/torch_patch/recsys_src_data.py eca/recsys/src/data/__init__.py
    !cp {CODE}/torch_patch/sasrec_pipeline.py eca/recsys/
    !test -f /content/ml-1m.zip || wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O /content/ml-1m.zip
    !unzip -o -q /content/ml-1m.zip -d /content/
    !cp /content/ml-1m/ratings.dat eca/recsys/data/movielens_1m/
    %cd /content/eca/recsys
    for s in need_train:
        W = f"{WORK}/split{s}"
        !EXT_SPLIT_SEED={s} python sasrec_pipeline.py --config config/main_movielens_1m.yaml --work {W} --seed {s} --extra_epochs 0
    %cd /content

for s in SPLITS:
    st, ca = SRC[s]
    if not os.path.exists(ca):
        print(f"разбиение {s}, собираю причинный контекст")
        run("run_context.py", f"{WORK}/split{s}", "context.log", EXT_STATES=st)

разбиение 0: состояния /content/drive/MyDrive/ext_work_v5/split0/sasrec_states.npz
разбиение 1: состояния /content/drive/MyDrive/ext_work_v5/split1/sasrec_states.npz
разбиение 2: состояния /content/drive/MyDrive/ext_work_v5/split2/sasrec_states.npz


## A2. Обучаемая память на состояниях SASRec

Для каждого разбиения скрипт сначала собирает кандидатов (около трёх минут), потом учит блок отдельно, контроль с состояниями и блок в смеси, по несколько минут на каждый. В логе видно, как падает ошибка на отложенных пользователях. После обучения он печатает таблицу на тесте.

Ноутбук заодно сверяет базу. NDCG@10 на последней позиции без контекста должен совпасть с тем, что SASRec показал на тесте при обучении. Если не совпадёт, ячейка напишет об этом, такой вывод надо прислать мне.

In [ ]:
for s in SPLITS:
    st, ca = SRC[s]
    W = f"{WORK}/split{s}"
    print(f"разбиение {s}")
    if not os.path.exists(f"{W}/results_learned_memory.json"):
        run("run_learned_memory.py", W, "learned_memory.stdout", EXT_STATES=st, EXT_CACHE=ca)
    summ = [p for p in (f"{W}/sasrec_summary.json", f"{OLD5}/split{s}/sasrec_summary.json") if os.path.exists(p)]
    if summ:
        b_sasrec = json.load(open(summ[0]))["pretrained"]["test"]
        b_numpy = json.load(open(f"{W}/results_learned_memory.json"))["info"]["base_ndcg_last"]
        print(f"    база SASRec {b_sasrec:.4f}, база в numpy {b_numpy:.4f}" +
              ("" if abs(b_sasrec - b_numpy) < 0.002 else "   НЕ СОВПАЛИ, пришли этот вывод"))
    show("summarize.py", W)

разбиение 0
    run_learned_memory.py готов за 622 с
    база SASRec 0.1099, база в numpy 0.1099
Результаты из папки /content/drive/MyDrive/ext_work_v7/split0

Обучаемая память переходов, сначала блок отдельно, потом гейт
  память 455421 переходов, кандидатов на запрос 200, таргет среди кандидатов у 0.576 тестовых запросов
  блок учился на 8473 ранних позициях валидационных пользователей, остановка по 1995
  тест, NDCG@10 по 5 позициям на пользователя, база 0.1062 (на последней позиции 0.1099), пользователей 805
    память без обучения                            0.1130  +0.0068 [+0.0026; +0.0111]
    блок отдельно, гейт lambda                     0.1155  +0.0093 [+0.0055; +0.0132]
    блок в смеси, гейт lambda                      0.1132  +0.0070 [+0.0038; +0.0103]
    блок отдельно, гейт по признакам               0.1184  +0.0122 [+0.0083; +0.0163]
    состояния, смесь в вероятностях                0.1097  +0.0035 [+0.0004; +0.0066]
    состояния, смесь в эмбеддинге как в статье     0

## A3. Все разбиения вместе

In [ ]:
show("combine_learned.py", WORK, *[f"{WORK}/split{s}" for s in SPLITS])

Обучаемая память, 3 разбиений: split0, split1, split2
  базы 0.1062, 0.1016, 0.0981
  память без обучения                            +0.0068 +0.0022 +0.0040   вместе +0.0043 [+0.0024; +0.0061]
  блок отдельно, гейт lambda                     +0.0093 +0.0010 +0.0053   вместе +0.0052 [+0.0034; +0.0070]
  блок в смеси, гейт lambda                      +0.0070 +0.0000 +0.0051   вместе +0.0041 [+0.0023; +0.0058]
  блок отдельно, гейт по признакам               +0.0122 +0.0013 +0.0063   вместе +0.0066 [+0.0047; +0.0085]
  состояния, смесь в вероятностях                +0.0035 +0.0000 +0.0010   вместе +0.0015 [+0.0003; +0.0027]
  состояния, смесь в эмбеддинге как в статье     +0.0037 +0.0000 +0.0017   вместе +0.0018 [+0.0007; +0.0029]
  сравнения вместе
    обучение блока против памяти без обучения      +0.0009 [-0.0002; +0.0020]
    дообучение в смеси против отдельного           -0.0012 [-0.0021; -0.0002]
    гейт по признакам против константы             +0.0014 [+0.0003; +0.0025]
    метки

# Часть B. Churn в постановке статьи

Здесь тот же блок на банковских данных авторов. Задача предсказать MCC следующей транзакции, разбиение по клиентам, энкодер наш MS-EMA, как в части B в v5 и v6. Контекст видит только события строго раньше дня запроса. Если в ext_work_v6 остались энкодер и кеш, ноутбук возьмёт их оттуда.

In [ ]:
def get_bank(n):
    bank_dir = f"{WORK}/bank_data"
    os.makedirs(bank_dir, exist_ok=True)
    dst = f"{bank_dir}/{n}.parquet"
    for cand in glob.glob(f"{ROOT}/ext_work_v*/bank_data/{n}.parquet") + glob.glob(f"/content/{n}.parquet"):
        if not os.path.exists(dst):
            shutil.copy(cand, dst)
    if os.path.exists(dst):
        return dst
    subprocess.run(["pip", "-q", "install", "-U", "gdown"])
    import inspect, gdown
    params = inspect.signature(gdown.download_folder).parameters
    if "skip_download" not in params:
        print("эта версия gdown не умеет получать список файлов без скачивания")
        return None
    kw = dict(id="1VFGL6ZOtnP7ErPyVlmQv62NJTXUs8ekY", skip_download=True, quiet=True)
    if "remaining_ok" in params:
        kw["remaining_ok"] = True
    try:
        listing = gdown.download_folder(**kw) or []
        paths = [(getattr(f, "id", None), getattr(f, "path", "") or getattr(f, "local_path", "")) for f in listing]
        hits = sorted([(i, p) for i, p in paths if os.path.basename(p) == f"{n}.parquet"], key=lambda x: "preprocessed" not in x[1])
        if hits:
            gdown.download(id=hits[0][0], output=dst, quiet=False)
        else:
            print(f"файла {n}.parquet в папке не видно, вот что там есть")
            for i, p in paths[:60]:
                print("   ", p)
    except Exception as err:
        print("автоматически скачать не вышло", repr(err)[:300])
    return dst if os.path.exists(dst) else None


if RUN_BANK:
    src = get_bank(BANK)
    if src is None:
        print("нет данных Churn. Скачай churn.parquet из папки preprocessed по ссылке",
              "https://drive.google.com/drive/folders/1VFGL6ZOtnP7ErPyVlmQv62NJTXUs8ekY,",
              "положи в ext_work_v7/bank_data на Drive и запусти ячейку снова")
    else:
        W = f"{WORK}/bank_{BANK}_split0"
        os.makedirs(W, exist_ok=True)
        env = dict(EXT_DATASET="bank", EXT_BANK_FILE=src, EXT_BANK_MAX_USERS=BANK_MAX_USERS,
                   EXT_SPLIT_SEED=0, EXT_STRICT_PAST=1)
        old = f"{OLD6}/bank_{BANK}_split0"
        if os.path.exists(f"{W}/cache_context.pkl"):
            cache = f"{W}/cache_context.pkl"
        elif os.path.exists(f"{old}/encoder_msema.npz") and os.path.exists(f"{old}/cache_context.pkl"):
            shutil.copy(f"{old}/encoder_msema.npz", W)
            cache = f"{old}/cache_context.pkl"
            print("беру энкодер и кеш из", old)
        else:
            if not os.path.exists(f"{W}/encoder_msema.npz"):
                run("run_encoder2.py", W, "encoder_train.log", **env)
            run("run_context.py", W, "context.log", **env)
            cache = f"{W}/cache_context.pkl"
        if not os.path.exists(f"{W}/results_learned_memory.json"):
            run("run_learned_memory.py", W, "learned_memory.stdout", EXT_CACHE=cache, **env)
        show("summarize.py", W)

беру энкодер и кеш из /content/drive/MyDrive/ext_work_v6/bank_churn_split0
    run_learned_memory.py готов за 854 с
Результаты из папки /content/drive/MyDrive/ext_work_v7/bank_churn_split0

Обучаемая память переходов, сначала блок отдельно, потом гейт
  память 340068 переходов, кандидатов на запрос 200, таргет среди кандидатов у 0.895 тестовых запросов
  блок учился на 15739 ранних позициях валидационных пользователей, остановка по 4010
  тест, NDCG@10 по 5 позициям на пользователя, база 0.5496 (на последней позиции 0.5388), пользователей 969
    память без обучения                            0.5502  +0.0005 [-0.0018; +0.0028]
    блок отдельно, гейт lambda                     0.5502  +0.0006 [-0.0008; +0.0019]
    блок в смеси, гейт lambda                      0.5500  +0.0004 [-0.0020; +0.0028]
    блок отдельно, гейт по признакам               0.5506  +0.0010 [-0.0005; +0.0025]
    состояния, смесь в вероятностях                0.5496  +0.0000 [+0.0000; +0.0000]
    состояния, смесь 

## Сводка и сохранение

Скопируй весь вывод следующей ячейки и пришли мне вместе с архивом из последней ячейки.

In [ ]:
for W in [f"{WORK}/split{s}" for s in SPLITS] + sorted(glob.glob(f"{WORK}/bank_*_split0")):
    if os.path.exists(f"{W}/results_learned_memory.json"):
        print("=" * 30, os.path.basename(W), "=" * 30)
        show("summarize.py", W)
show("combine_learned.py", WORK, *[f"{WORK}/split{s}" for s in SPLITS])

============================== split0 ==============================
Результаты из папки /content/drive/MyDrive/ext_work_v7/split0

Обучаемая память переходов, сначала блок отдельно, потом гейт
  память 455421 переходов, кандидатов на запрос 200, таргет среди кандидатов у 0.576 тестовых запросов
  блок учился на 8473 ранних позициях валидационных пользователей, остановка по 1995
  тест, NDCG@10 по 5 позициям на пользователя, база 0.1062 (на последней позиции 0.1099), пользователей 805
    память без обучения                            0.1130  +0.0068 [+0.0026; +0.0111]
    блок отдельно, гейт lambda                     0.1155  +0.0093 [+0.0055; +0.0132]
    блок в смеси, гейт lambda                      0.1132  +0.0070 [+0.0038; +0.0103]
    блок отдельно, гейт по признакам               0.1184  +0.0122 [+0.0083; +0.0163]
    состояния, смесь в вероятностях                0.1097  +0.0035 [+0.0004; +0.0066]
    состояния, смесь в эмбеддинге как в статье     0.1099  +0.0037 [+0.0011; +0.

In [ ]:
!cd {WORK} && rm -f /content/results_v7.zip && zip -q -r /content/results_v7.zip . -x "*.npz" "*.pkl" "*.pth" "*.zip" "*.parquet"
if USE_DRIVE:
    shutil.copy("/content/results_v7.zip", WORK)
from google.colab import files
files.download("/content/results_v7.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Как читать результаты

Главная строка это блок отдельно с гейтом lambda против памяти без обучения. Если она положительна, предложение Зайцева работает, обученный отдельно блок лучше выбирает, на чьи переходы смотреть. Строка про дообучение в смеси показывает, есть ли смысл учить блок сразу на то, где SASRec ошибается. Строка метки против состояний это ключевой контроль. Блок с одинаковым обучением и одинаковым гейтом сравнивается с вариантом, где значения это состояния, как в статье. Если метки заметно лучше, значит дело именно в том, что агрегируется, а не в обучении.

В настройках блока есть recency, это выученный вес свежести. Если он заметно больше нуля, блок предпочитает недавние переходы, и тогда внешний контекст несёт что-то про текущий момент. Если он около нуля, свежесть не важна, и блок работает за счёт похожих ситуаций.

Строка таргет среди кандидатов показывает потолок. Блок только переранжирует 200 кандидатов, поэтому там, где правильного следующего события среди них нет, он помочь не может. Строка про блоки сами по себе показывает, насколько хорошо каждый блок предсказывает без SASRec.

Если что-то пошло не так, ячейка покажет хвост лога упавшего шага, его и пришли.